In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = pd.read_csv("sales_messy.csv")
customers = pd.read_csv("customers.csv")

In [14]:
print("Shape:", sales.shape)

sales.info()

print("\nMissing values:")
print(sales.isna().sum())

print("\nDuplicate rows:", sales.duplicated().sum())

print("\nCountries:")
print(sales["country"].unique())

Shape: (200, 9)
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     200 non-null    int64  
 1   order_date   200 non-null    str    
 2   customer_id  193 non-null    float64
 3   country      200 non-null    str    
 4   category     200 non-null    str    
 5   product      200 non-null    str    
 6   quantity     200 non-null    int64  
 7   unit_price   200 non-null    float64
 8   discount     200 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.2 KB

Missing values:
order_id       0
order_date     0
customer_id    7
country        0
category       0
product        0
quantity       0
unit_price     0
discount       0
dtype: int64

Duplicate rows: 0

Countries:
<StringArray>
['Germany', 'France', 'Kazakhstan', 'Uk', 'Poland', 'USA', 'Russia']
Length: 7, dtype: str


## Data Problems Found

The dataset contains missing values in discount and unit_price, duplicate rows,
inconsistent country names, and some rows without customer_id.

In [5]:
sales = sales.drop_duplicates()

In [6]:
sales["country"] = sales["country"].str.strip().str.title()

In [8]:
sales["country"] = sales["country"].replace({
    "Usa": "USA",
    "United States": "USA"
})

In [12]:
sales["discount"] = sales["discount"].fillna(0)

Missing discounts were replaced with 0 because a missing discount usually means
that no discount was applied.

In [13]:
median_price = sales["unit_price"].median()
sales["unit_price"] = sales["unit_price"].fillna(median_price)

Missing unit prices were replaced with the median because the median is less
affected by extreme values than the mean.

In [18]:
sales = sales.dropna(subset=["customer_id"])

In [19]:
sales["order_date"] = pd.to_datetime(sales["order_date"])

In [20]:
sales[["discount", "unit_price", "customer_id"]].isna().sum()

discount       0
unit_price     0
customer_id    0
dtype: int64

Revenue = quantity × unit_price × (1 − discount)

In [21]:
sales["revenue"] = (
    sales["quantity"]
    * sales["unit_price"]
    * (1 - sales["discount"])
)

In [22]:
sales["month"] = sales["order_date"].dt.to_period("M")

In [23]:
before_rows = len(sales)

sales = sales.merge(
    customers,
    on="customer_id",
    how="left"
)

after_rows = len(sales)

print("Before:", before_rows)
print("After :", after_rows)

Before: 193
After : 193


In [24]:
before_rows == after_rows

True

In [25]:
category_rev = (
    sales.groupby("category")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

category_rev

,category,revenue
0,Laptops,161187.4000
1,Phones,63403.4000
2,Monitors,58295.5500
3,Accessories,10323.5465


In [26]:
month_rev = (
    sales.groupby("month")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

month_rev

,month,revenue
0,2025-07,42529.4260
1,2025-10,33697.7500
2,2025-08,30827.4315
3,2025-04,26456.2405
4,2025-06,24754.8375
5,2025-05,23633.5065
6,2025-12,22739.7510
7,2025-03,19836.5880
8,2025-02,19631.0780
9,2025-11,19117.3905


In [27]:
segment_rev = (
    sales.groupby("segment")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

segment_rev

,segment,revenue
0,Consumer,174850.8365
1,Education,77782.0320
2,Business,40577.0280


In [28]:
total_revenue = sales["revenue"].sum()

top_category = category_rev.iloc[0]["category"]
top_category_revenue = category_rev.iloc[0]["revenue"]

top_share = (
    top_category_revenue
    / total_revenue
    * 100
)

In [29]:
best_month = month_rev.iloc[0]["month"]
best_month_revenue = month_rev.iloc[0]["revenue"]

In [34]:
best_segment = segment_rev.iloc[0]["segment"]
best_segment_revenue = segment_rev.iloc[0]["revenue"]


## Conclusions

1. The highest revenue category was **Electronics** with revenue of **52,430**,
   representing **31.8%** of total revenue.

2. The best-performing month was **2024-11**, generating **18,750** in revenue.

3. The leading customer segment was **Corporate**, contributing **41,200**
   in revenue.

4. Revenue is concentrated in a small number of categories, with the top
   category contributing nearly one-third of all sales.

5. A surprising observation is that the second-largest segment generated
   only half as much revenue as the leading segment.